# 09. 파이썬 기초 - numpy

`numpy` 는 **숫자 계산 전용 라이브러리** 입니다.

pandas 의 DataFrame 은 사실 **numpy 배열 위에 이름표를 붙인 것** 이라,
numpy 를 알면 pandas 가 왜 그렇게 동작하는지 이해됩니다.

**다루는 내용**
1. ndarray — 리스트와 무엇이 다른가
2. 벡터 연산 — 반복문 없이 계산
3. 브로드캐스팅
4. 불리언 마스크 — pandas 필터링의 원리
5. 집계와 `axis`
6. 결측치 `np.nan`
7. pandas 와의 관계

In [1]:
import numpy as np
import pandas as pd

print('numpy', np.__version__)

numpy 1.26.4


## 1. ndarray — 리스트와 무엇이 다른가

- **리스트**: 아무 타입이나 담을 수 있음. 계산하려면 반복문 필요
- **ndarray**: **같은 타입만** 담음. 대신 **전체를 한 번에 계산**

`04pandas기초` 에서 `range()` 와 `np.arange()` 를 비교했던 이유가 여기 있습니다.

In [2]:
py_list = [1, 2, 3, 4, 5]
arr = np.array(py_list)

print('리스트 :', py_list, type(py_list))
print('배열   :', arr, type(arr))

print()
print('shape (모양) :', arr.shape)    # (5,) → 1차원 5개
print('dtype (타입) :', arr.dtype)    # int64 → 모든 값이 정수
print('ndim  (차원) :', arr.ndim)

리스트 : [1, 2, 3, 4, 5] <class 'list'>
배열   : [1 2 3 4 5] <class 'numpy.ndarray'>

shape (모양) : (5,)
dtype (타입) : int32
ndim  (차원) : 1


In [3]:
# 2차원 배열 = 표. DataFrame 의 실체가 이것이다
mat = np.array([
    [1, 2, 3],
    [4, 5, 6],
])
print(mat)
print('\nshape:', mat.shape, '→ 2행 3열')

print()
# 자주 쓰는 생성 함수
print('arange(0,10,2):', np.arange(0, 10, 2))     # 0부터 10 전까지 2씩
print('zeros(3)      :', np.zeros(3))              # 0 으로 채운 배열
print('linspace(0,1,5):', np.linspace(0, 1, 5))    # 0~1 을 5등분

[[1 2 3]
 [4 5 6]]

shape: (2, 3) → 2행 3열

arange(0,10,2): [0 2 4 6 8]
zeros(3)      : [0. 0. 0.]
linspace(0,1,5): [0.   0.25 0.5  0.75 1.  ]


## 2. 벡터 연산 — 반복문 없이 계산 ⭐

numpy 의 핵심입니다. **배열에 연산자를 쓰면 모든 원소에 한 번에 적용** 됩니다.

`df['price'] * 1.1` 이 동작하는 이유가 바로 이것입니다.

In [4]:
prices = np.array([25000, 30000, 28000, 33000])

# 리스트였다면 반복문이 필요하다
py_result = [p * 1.1 for p in [25000, 30000, 28000, 33000]]
print('리스트 + 반복문:', py_result)

# 배열은 그냥 곱하면 된다
print('배열 벡터 연산 :', prices * 1.1)

print()
print('덧셈  :', prices + 1000)
print('나눗셈:', prices / 1000)
print('비교  :', prices > 28000)   # 결과도 배열 (True/False)

리스트 + 반복문: [27500.000000000004, 33000.0, 30800.000000000004, 36300.0]
배열 벡터 연산 : [27500. 33000. 30800. 36300.]

덧셈  : [26000 31000 29000 34000]
나눗셈: [25. 30. 28. 33.]
비교  : [False  True False  True]


In [5]:
# 배열끼리의 연산은 '같은 자리끼리' 계산된다
qty = np.array([2, 1, 3, 1])

print('가격:', prices)
print('수량:', qty)
print('금액:', prices * qty)        # 자리별 곱셈
print('총액:', (prices * qty).sum())

가격: [25000 30000 28000 33000]
수량: [2 1 3 1]
금액: [50000 30000 84000 33000]
총액: 197000


## 3. 브로드캐스팅

모양이 다른 배열끼리 연산할 때, numpy 가 **자동으로 모양을 맞춰줍니다.**

`prices * 1.1` 에서 `1.1` 이 4개로 늘어난 것도 브로드캐스팅입니다.

In [6]:
mat = np.array([
    [1, 2, 3],
    [4, 5, 6],
])

# (2,3) 배열 + 스칼라 → 모든 원소에 더해짐
print('mat + 10:')
print(mat + 10)

print()
# (2,3) 배열 + (3,) 배열 → 각 행에 더해짐
row = np.array([100, 200, 300])
print('mat + [100,200,300]:')
print(mat + row)

mat + 10:
[[11 12 13]
 [14 15 16]]

mat + [100,200,300]:
[[101 202 303]
 [104 205 306]]


## 4. 불리언 마스크 — pandas 필터링의 원리 ⭐

`df[df['price'] > 28000]` 이 왜 동작하는지 여기서 알 수 있습니다.

1. 비교 연산 → **True/False 배열** 이 만들어짐
2. 그 배열을 대괄호에 넣으면 → **True 자리의 값만** 남음

In [7]:
prices = np.array([25000, 30000, 28000, 33000])

# 1단계: 조건 → True/False 배열
mask = prices > 28000
print('마스크:', mask)

# 2단계: 마스크로 골라내기
print('선택된 값:', prices[mask])

print()
# 한 줄로 (pandas 문법과 똑같은 모양)
print('한 줄로  :', prices[prices > 28000])

print()
# 조건 결합: & (그리고) | (또는) — 각 조건을 반드시 괄호로 감쌀 것
print('28000 초과 & 33000 미만:', prices[(prices > 28000) & (prices < 33000)])

마스크: [False  True False  True]
선택된 값: [30000 33000]

한 줄로  : [30000 33000]

28000 초과 & 33000 미만: [30000]


In [8]:
# np.where(조건, 참일때, 거짓일때) — 조건에 따라 값 바꾸기
grade = np.where(prices >= 30000, '고가', '저가')
print('가격:', prices)
print('등급:', grade)

# pandas 에서도 똑같이 쓴다
df = pd.DataFrame({'price': prices})
df['grade'] = np.where(df['price'] >= 30000, '고가', '저가')
print()
print(df)

가격: [25000 30000 28000 33000]
등급: ['저가' '고가' '저가' '고가']

   price grade
0  25000    저가
1  30000    고가
2  28000    저가
3  33000    고가


## 5. 집계와 `axis` ⭐

2차원 배열에서 `axis` 는 **어느 방향으로 계산할지** 지정합니다.

| axis | 방향 | 결과 |
|---|---|---|
| `axis=0` | **세로**(행을 따라 내려가며) | 열별 결과 |
| `axis=1` | **가로**(열을 따라 옆으로) | 행별 결과 |

> 헷갈릴 때: **"axis=0 은 행이 사라진다, axis=1 은 열이 사라진다"** 로 기억

In [9]:
scores = np.array([
    [90, 80, 70],    # 학생1 의 국영수
    [60, 95, 85],    # 학생2
])
print(scores, '\nshape:', scores.shape)

print()
print('전체 평균        :', scores.mean())
print('axis=0 (과목별)  :', scores.mean(axis=0))   # 열별 → 과목 평균
print('axis=1 (학생별)  :', scores.mean(axis=1))   # 행별 → 학생 평균

[[90 80 70]
 [60 95 85]] 
shape: (2, 3)

전체 평균        : 80.0
axis=0 (과목별)  : [75.  87.5 77.5]
axis=1 (학생별)  : [80. 80.]


In [10]:
# 자주 쓰는 집계 함수
print('합계:', scores.sum())
print('최대:', scores.max(), '/ 최소:', scores.min())
print('표준편차:', round(scores.std(), 2))

print()
# argmax: '가장 큰 값' 이 아니라 '가장 큰 값의 위치(인덱스)'
print('학생별 최고점 과목 위치:', scores.argmax(axis=1))
print('→ 0=국어, 1=영어, 2=수학')

합계: 480
최대: 95 / 최소: 60
표준편차: 11.9

학생별 최고점 과목 위치: [0 1]
→ 0=국어, 1=영어, 2=수학


## 6. 결측치 `np.nan`

`np.nan` 은 "값이 없음" 을 뜻하는 특별한 숫자입니다. pandas 의 빈 칸이 바로 이것입니다.

**주의: `nan` 이 하나라도 섞이면 일반 집계 결과가 전부 `nan` 이 됩니다.**

In [11]:
heights = np.array([172, np.nan, 168, 180])
print('배열:', heights)

print()
print('mean()    :', heights.mean())      # nan! 하나만 있어도 전체가 nan
print('nanmean() :', np.nanmean(heights))  # nan 을 무시하고 계산

print()
# nan 은 자기 자신과도 같지 않다 → == 로 못 찾는다
print('nan == nan :', np.nan == np.nan)
print('isnan()    :', np.isnan(heights))   # 이 함수로 찾아야 한다

배열: [172.  nan 168. 180.]

mean()    : nan
nanmean() : 173.33333333333334

nan == nan : False
isnan()    : [False  True False False]


## 7. pandas 와의 관계

DataFrame 의 각 열은 **ndarray 에 인덱스와 이름을 붙인 것** 입니다.
그래서 pandas 에서 numpy 함수를 그대로 쓸 수 있습니다.

In [12]:
df = pd.DataFrame({
    'name':   ['A', 'B', 'C', 'D'],
    'height': [172, 165, 168, 180],
    'weight': [55, 50, 58, 70],
})

# .values 또는 .to_numpy() 로 numpy 배열을 꺼낼 수 있다
print('Series 의 내부:', df['height'].to_numpy(), type(df['height'].to_numpy()))

print()
# numpy 함수를 pandas 열에 그대로 적용
df['bmi'] = (df['weight'] / (df['height'] / 100) ** 2).round(1)
df['grade'] = np.where(df['bmi'] >= 22, '높음', '보통')
print(df)

Series 의 내부: [172 165 168 180] <class 'numpy.ndarray'>

  name  height  weight   bmi grade
0    A     172      55  18.6    보통
1    B     165      50  18.4    보통
2    C     168      58  20.5    보통
3    D     180      70  21.6    보통


In [13]:
# 실전: 인구현황 데이터에 numpy 연산 적용
pop = pd.read_csv('../data/인구현황.csv')
pop = pop[pop['행정기관'] != '전국']       # '전국' 합계 행 제외

# 남녀 비율을 직접 계산 (벡터 연산 — 반복문 없음)
pop['남자비율'] = (pop['남자 인구수'] / pop['총인구수'] * 100).round(2)

print(pop[['행정기관', '총인구수', '남자비율']].head(5).to_string(index=False))

print()
print('전국 평균 남자비율:', round(pop['남자비율'].mean(), 2), '%')
print('총인구 합계       :', f"{pop['총인구수'].sum():,}")

 행정기관    총인구수  남자비율
서울특별시 9331828 48.28
부산광역시 3266598 48.67
대구광역시 2363629 49.06
인천광역시 3021010 49.96
광주광역시 1408422 49.36

전국 평균 남자비율: 50.02 %
총인구 합계       : 51,217,221


## 정리

| 개념 | 문법 | 왜 중요한가 |
|---|---|---|
| 벡터 연산 | `arr * 1.1` | 반복문 없이 전체 계산 → `df['price'] * 1.1` 의 원리 |
| 불리언 마스크 | `arr[arr > 10]` | `df[df['price'] > 10]` 의 원리 |
| 조건 치환 | `np.where(조건, A, B)` | 파생 열 만들기 |
| axis | `axis=0`(열별) / `axis=1`(행별) | 집계 방향 지정 |
| 결측치 | `np.nan`, `np.nanmean()`, `np.isnan()` | nan 하나로 전체가 nan 이 되는 것 방지 |

**꼭 기억할 것**
- 조건을 결합할 때는 `and`/`or` 가 아니라 `&`/`|` 를 쓰고, **각 조건을 괄호로 감싼다**
- `nan` 은 `==` 로 못 찾는다. `np.isnan()` 또는 pandas 의 `isna()` 를 쓴다

다음: `10python_basic_datetime.ipynb` (날짜와 시간 다루기)